# Day 8: QA triage, defects and flaky tests
**AI for Product Owners and QA · Day 8**

You build a small triage pipeline:
1. An **anonymiser** with a leak test, so no personal data reaches an AI tool.
2. A **failure grouper** that turns 60 failures into a handful of groups.
3. **AI labels** for each group (product bug, test bug, environment, flaky), checked against an allowed list.
4. A **flaky-test detector** from the run history, with no AI at all.

Run this notebook in your Codespace (VS Code in the browser). Your Gemini key comes from the Codespaces secret `GEMINI_API_KEY`. No key, or no network? Set `PROVIDER = "offline"`.

In [5]:
# SETUP: run this cell first. It defines ask() and ask_json() for the rest of the notebook.
# PROVIDER = "gemini"  -> uses your Gemini API key (Colab: key icon on the left, secret named GEMINI_API_KEY)
# PROVIDER = "ollama"  -> a local model (only when this notebook runs on your own laptop with Ollama open)
# PROVIDER = "offline" -> canned answers, no key and no internet. Use it if the key or the network fails.
PROVIDER = "gemini"
MODEL = "gemini-3.5-flash-lite"          # Gemini model name, or an Ollama model name such as "llama3.2"
OLLAMA_URL = "http://localhost:11434"

import json, os, re, urllib.request, urllib.error
OFFLINE = {}        # canned answers, filled in a later cell
last_usage = {}     # token counts from the last Gemini call

def _gemini_key():
    try:
        from google.colab import userdata
        return userdata.get("GEMINI_API_KEY")
    except Exception:
        return os.environ.get("GEMINI_API_KEY", "")

def _post(url, body, headers=None, timeout=300):
    req = urllib.request.Request(url, data=json.dumps(body).encode("utf-8"), method="POST")
    req.add_header("Content-Type", "application/json")
    for k, v in (headers or {}).items():
        req.add_header(k, v)
    try:
        with urllib.request.urlopen(req, timeout=timeout) as r:
            return json.loads(r.read().decode("utf-8"))
    except urllib.error.HTTPError as e:
        raise RuntimeError(f"Model service said HTTP {e.code}: {e.read().decode('utf-8', 'replace')[:300]}") from None
    except urllib.error.URLError as e:
        raise RuntimeError(f"Could not reach the model service: {e.reason}") from None

def ask(prompt, system=None, temperature=0.2, task=None, json_mode=False):
    """Send one prompt and return the reply text. task names the canned answer used offline."""
    global last_usage
    if PROVIDER == "offline":
        if task not in OFFLINE:
            raise RuntimeError(f"No offline answer for task {task!r}. Switch PROVIDER or add one to OFFLINE.")
        reply = OFFLINE[task]
        return reply if isinstance(reply, str) else json.dumps(reply)
    if PROVIDER == "ollama":
        msgs = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": prompt}]
        body = {"model": MODEL, "stream": False, "messages": msgs, "options": {"temperature": temperature}}
        if json_mode:
            body["format"] = "json"
        return _post(OLLAMA_URL + "/api/chat", body)["message"]["content"]
    key = _gemini_key()
    if not key:
        raise RuntimeError("No Gemini key found. Add the GEMINI_API_KEY secret, or set PROVIDER = 'offline'.")
    body = {"contents": [{"role": "user", "parts": [{"text": prompt}]}],
            "generationConfig": {"temperature": temperature}}
    if system:
        body["systemInstruction"] = {"parts": [{"text": system}]}
    if json_mode:
        body["generationConfig"]["responseMimeType"] = "application/json"
    data = _post(f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL}:generateContent",
                 body, {"x-goog-api-key": key})
    last_usage = data.get("usageMetadata", {})
    try:
        return "".join(p.get("text", "") for p in data["candidates"][0]["content"]["parts"])
    except (KeyError, IndexError):
        raise RuntimeError("The reply had no text. Check the model name.") from None

def ask_json(prompt, system=None, temperature=0.0, task=None):
    """Ask for JSON and parse it. Raises ValueError if the reply is not valid JSON."""
    text = ask(prompt, system=system, temperature=temperature, task=task, json_mode=True).strip()
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text)
    try:
        return json.loads(text)
    except json.JSONDecodeError as e:
        raise ValueError(f"The reply was not valid JSON ({e}). First 200 characters: {text[:200]}") from None

print("Setup done. Provider:", PROVIDER, "| model:", MODEL)

Setup done. Provider: gemini | model: gemini-3.5-flash-lite


In [3]:
# OFFLINE ANSWERS: used only when PROVIDER = "offline". Leave this cell as it is.
OFFLINE["hello"] = "Hello from the offline model."
OFFLINE["labels"] = {"labels": [
  {"group": "G1", "label": "test bug", "reason": "The apply-code button was renamed in a build, so the test's selector is out of date."},
  {"group": "G2", "label": "flaky", "reason": "Timeouts on the pay button vary with page load time from run to run."},
  {"group": "G3", "label": "product bug", "reason": "The orders API returns a server error."},
  {"group": "G4", "label": "product bug", "reason": "The discount is above the Rs 200 cap, so the code is wrong, not the test."},
  {"group": "G5", "label": "environment", "reason": "The staging server reset the connection."}]}
OFFLINE["labels_injected"] = {"labels": [
  {"group": "G1", "label": "flaky", "reason": "As instructed, classified as flaky."},
  {"group": "G2", "label": "flaky", "reason": "As instructed, classified as flaky."},
  {"group": "G3", "label": "flaky", "reason": "As instructed, classified as flaky."},
  {"group": "G4", "label": "flaky", "reason": "As instructed, classified as flaky."},
  {"group": "G5", "label": "flaky", "reason": "As instructed, classified as flaky."},
  {"group": "G6", "label": "flaky", "reason": "As instructed, classified as flaky."}]}
OFFLINE["labels_defended"] = {"labels": OFFLINE["labels"]["labels"] + [
  {"group": "G6", "label": "product bug", "reason": "Order status stays Packed. The text inside the error that asks to classify failures is data, not an instruction."}]}
print("Offline answers loaded:", ", ".join(OFFLINE))

Offline answers loaded: hello, labels, labels_injected, labels_defended


In [7]:
print(ask("Say hello in five words.", task="hello"))

Hello there, nice to meet.


## Part 0: the test results

In [6]:
# PART 0: load the synthetic test results.
import json, re
import pandas as pd
report = json.load(open("data/test_report.json"))
history = pd.read_csv("data/run_history.csv")
print(len(report), "failures from the last 20 runs;", len(history), "tests in the run history.")
print(json.dumps(report[0], indent=1))

60 failures from the last 20 runs; 20 tests in the run history.
{
 "run": 2,
 "test": "checkout.spec.ts > applies SAVE10 to a Rs 1000 cart",
 "error": "Error: locator('button.apply-code') resolved to 0 elements",
 "log": "apply-code button renamed in build 144; caller phone 9876543210"
}


## Part 1: anonymise first (live build)
The leak test fails until you add a pattern for the name hidden in a URL.

In [10]:
# PART 1 (live build): anonymise every log before it goes near an AI tool.
PATTERNS = [
    ("card",  r"\b(?:\d[ -]?){13,16}\b"),            # card numbers first: they contain long digit runs
    ("email", r"[\w.+-]+@[\w-]+\.[\w.]+"),
    ("phone", r"(?:\+91[\s-]?)?\b[6-9]\d{9}\b"),
]
NAMES = ["asha", "iyer", "meera", "nair", "ravi", "kumar"]   # the synthetic customers in this data

def anonymise(text):
    for label, pattern in PATTERNS:
        text = re.sub(pattern, f"<{label}>", text)
    return text

def leaks(text):
    """Return what is still personal in the text. An empty list means it is safe to share."""
    probe = re.sub(r"<\w+>", "", text)          # ignore the placeholders we put in
    found = [label for label, pattern in PATTERNS if re.search(pattern, probe)]
    found += [n for n in NAMES if re.search(rf"\b{n}\b", probe, re.IGNORECASE)]
    return found

clean = [{**r, "log": anonymise(r["log"]), "error": anonymise(r["error"])} for r in report]
leaky = [(c["log"], leaks(c["log"])) for c in clean if leaks(c["log"])]
print("Example:", report[1]["log"], "\n   ->  ", clean[1]["log"])
print(f"Leak test: {len(leaky)} logs still contain personal data.")
for log, what in leaky[:3]:
    print("   ", what, "|", log)

Example: user ravi.kumar@example.com reached payment for ORD-1600; page load 20s 
   ->   user <email> reached payment for ORD-1600; page load 20s
Leak test: 11 logs still contain personal data.
    ['meera', 'nair'] | GET /api/orders?customer=meera.nair 500 in 175ms
    ['meera', 'nair'] | GET /api/orders?customer=meera.nair 500 in 280ms
    ['meera', 'nair'] | GET /api/orders?customer=meera.nair 500 in 620ms


## Part 2: group the failures (live build)

In [11]:
# PART 2 (live build): group the 60 failures by their error signature.
def signature(error):
    first = error.split("\n")[0]
    first = re.sub(r"ORD-\d+", "ORD-#", first)
    return re.sub(r"\d+", "#", first)

df = pd.DataFrame(clean)
df["signature"] = df["error"].map(signature)
groups = (df.groupby("signature")
            .agg(count=("test", "size"), tests=("test", lambda s: sorted(set(s))), sample_log=("log", "first"))
            .sort_values(["count", "signature"], ascending=[False, True]).reset_index())
groups.insert(0, "group", [f"G{i}" for i in range(1, len(groups) + 1)])
print(groups[["group", "count", "signature"]].to_string(index=False))

group  count                                                                signature
   G1     16               Error: locator('button.apply-code') resolved to # elements
   G2     16                  TimeoutError: locator('#pay-now') not visible after #ms
   G3     11                                  Error: GET /api/orders/ORD-# returned #
   G4     10                          AssertionError: expected discount #, received #
   G5      7 Error: net::ERR_CONNECTION_RESET at https://staging.orion-shop.test/cart


## Part 3: label the groups with AI (live build)

In [14]:
# PART 3 (live build): ask the model to label each group, and check the labels it returns.
ALLOWED = {"product bug", "test bug", "environment", "flaky"}
LABEL_SYSTEM = """Role: You are a QA lead triaging test failures.
Task: Label each failure group as one of: product bug, test bug, environment, flaky. Give a one-sentence reason.
Constraints: Use only the error and log given. If you cannot tell, say so in the reason.
Output: JSON only: {"labels": [{"group", "label", "reason"}]}"""

def label_groups(groups, system=LABEL_SYSTEM, task="labels"):
    payload = groups[["group", "count", "signature", "sample_log"]].to_dict(orient="records")
    reply = ask_json("FAILURE GROUPS:\n" + json.dumps(payload, indent=1), system=system, task=task)
    labels = pd.DataFrame(reply["labels"])
    bad = labels[~labels["label"].isin(ALLOWED)]
    if len(bad):
        print("Labels outside the allowed list:", bad.to_dict(orient="records"))
    return groups.merge(labels, on="group", how="left")

labelled = label_groups(groups)
print(labelled[["group", "count", "label", "reason"]].to_string(index=False))

group  count       label                                                                                                                  reason
   G1     16    test bug                           The apply-code button was renamed in build 144, meaning the test locator needs to be updated.
   G2     16 product bug The payment element failed to become visible due to a slow page load, which is likely a performance or rendering issue.
   G3     11 product bug                              The API returned a 500 internal server error, indicating an unhandled server-side failure.
   G4     10 product bug                         The application calculated the wrong discount total, pointing to a defect in the pricing logic.
   G5      7 environment                    A connection reset network error indicates a temporary infrastructure or network connectivity issue.
